# QuantumDLBlocks: JAX + Equinox + Haliax + Optax + Levanter (`qdlblocks_jax`)

This notebook demonstrates the optional `[jax]` extra (`qdlblocks_jax`), featuring:
- **Named-Axis Quantum Born GPT (`HaxQuantumBornGPT`)** using `haliax.NamedArray` axes (`Vocab`, `Seq`, `Seed`, `Hilbert`, `Heads`, `HeadDim`, `MlpDim`, `Experts`).
- **$O(T)$ Memory Flash-Born Attention in JAX (`flash_born_attention_jax`)** with custom backward VJP (`@jax.custom_vjp`) and Pallas/Scan backends.
- **Optax Stiefel-Manifold Muon (`scale_by_stiefel_muon` & `build_quantum_optax_optimizer`)** with Continuous Riemannian Stiefel Penalty.
- **Multi-Qubit Tensor Factorization (`factorize_into_qubits`) & Bipartite Entanglement Entropy (`bipartite_entanglement_entropy`)** across subsystem partitions.

In [ ]:
# Install QuantumDLBlocks with [jax] extra (if running on Google Colab)
# !pip install -q "git+https://github.com/festivity/QuantumDLBlocks.git#egg=qdlblocks[jax]"
import equinox as eqx
import jax
import jax.numpy as jnp
import qdlblocks_jax as qjax
import haliax as hax

print("JAX devices:", jax.devices())

## 1. Build Named-Axis 6-Qubit `HaxQuantumBornGPT` with Sparse Born MoE
With `embed_dim=64`, the Hilbert space $\mathcal{H} \cong (\mathbb{C}^2)^{\otimes 6}$ corresponds to a 6-qubit register that can be factorized into named axes `(qubit_0, ..., qubit_5)`.

In [ ]:
key = jax.random.PRNGKey(42)
cfg = qjax.HaxQuantumBornLMConfig(
    vocab_size=128,
    seq_len=32,
    seed_dim=16,
    embed_dim=64,  # 2^6 = 64 (6 qubits)
    num_heads=4,
    mlp_dim=128,
    num_layers=2,
    num_experts=4,
    top_k_experts=2,
    use_flash_born=True,
)
model = cfg.build(key=key)

Batch = hax.Axis("batch", 8)
Seq = model.Seq
base_seq = jnp.arange(Batch.size * Seq.size, dtype=jnp.int32).reshape(Batch.size, Seq.size) % 16
tokens = hax.NamedArray(base_seq, (Batch, Seq))
targets = hax.NamedArray((base_seq + 1) % 16, (Batch, Seq))

psi = model.encode(tokens)
psi_qubits = model.factorize_into_qubits(psi)
print("Encoded state axes   :", psi.axes)
print("Factorized qubit axes:", psi_qubits.axes)

## 2. Train with Optax `scale_by_stiefel_muon` & Track Bipartite Entanglement Entropy

In [ ]:
tx = qjax.build_quantum_optax_optimizer(model, muon_lr=0.02, adamw_lr=3e-3, stiefel_penalty=0.05)
opt_state = tx.init(eqx.filter(model, eqx.is_inexact_array))

@eqx.filter_jit
def train_step(m, state):
    @eqx.filter_value_and_grad
    def loss_fn(mod):
        _, loss = mod(tokens, targets)
        return loss
    loss_val, grads = loss_fn(m)
    params = eqx.filter(m, eqx.is_inexact_array)
    updates, new_state = tx.update(grads, state, params)
    return eqx.apply_updates(m, updates), new_state, loss_val

for step in range(1, 16):
    model, opt_state, loss_val = train_step(model, opt_state)
    if step in (1, 5, 10, 15):
        iso_err = float(qjax.stiefel_isometry_error(model.blocks[0].attn.w_q.array))
        psi_cur = model.encode(tokens)
        ent = float(jnp.mean(qjax.bipartite_entanglement_entropy(psi_cur.array, num_qubits=6, keep_qubits=(0, 1, 2))))
        print(f"Step {step:02d} | Loss: {float(loss_val):.4f} | Stiefel Error: {iso_err:.5f} | 3+3 Qubit S(rho_A): {ent:.4f} nats")